# ElevenLabs API Colab Launcher

This notebook is a thin launcher. The canonical workflow source of truth is `elevenlabs_api.py` in this repository.

By default, it fetches files from `main`. Set `GITHUB_REF` to a commit SHA to pin an exact version.


In [ ]:
import pathlib
import subprocess
import urllib.request

GITHUB_OWNER = "Just9120"
GITHUB_REPO = "Elevenlabs-API"
GITHUB_REF = "main"  # change to a commit SHA to pin

RAW_BASE = f"https://raw.githubusercontent.com/{GITHUB_OWNER}/{GITHUB_REPO}/{GITHUB_REF}"
REQ_URL = f"{RAW_BASE}/requirements-colab.txt"
SRC_URL = f"{RAW_BASE}/elevenlabs_api.py"

print(f"Using GitHub ref: {GITHUB_REF}")
print(f"requirements-colab.txt URL: {REQ_URL}")
print(f"elevenlabs_api.py URL: {SRC_URL}")

workdir = pathlib.Path('/content')
req_path = workdir / 'requirements-colab.txt'
src_path = workdir / 'elevenlabs_api.py'

urllib.request.urlretrieve(REQ_URL, req_path)
urllib.request.urlretrieve(SRC_URL, src_path)

subprocess.run(['python', '-m', 'pip', 'install', '-r', str(req_path)], check=True)

source = src_path.read_text(encoding='utf-8')
lines = source.splitlines()
sanitized_lines = []
removed_known_pip_line = False
expected_pip_markers = (
    'google-api-python-client',
    'google-auth-httplib2',
    'google-auth-oauthlib',
    'requests',
    'ipywidgets',
)
for idx, line in enumerate(lines, start=1):
    stripped = line.lstrip()
    if stripped.startswith('!') or stripped.startswith('%'):
        is_expected_top_level_pip = (
            stripped.startswith('!pip ')
            and idx <= 20
            and any(marker in stripped for marker in expected_pip_markers)
        )
        if is_expected_top_level_pip and not removed_known_pip_line:
            removed_known_pip_line = True
            print(f"Removed known Colab dependency install line at {idx}: {stripped}")
            continue
        if stripped.startswith('!pip '):
            raise RuntimeError(
                f"Unexpected !pip line in elevenlabs_api.py at line {idx}: {line!r}. \
Only the known top-level Colab dependency install line is allowed."
            )
        raise RuntimeError(
            f"Unsupported notebook magic/shell line in elevenlabs_api.py at line {idx}: {line!r}. \
Refuse to guess. Please sanitize upstream file or run directly in Colab."
        )
    sanitized_lines.append(line)

exec_globals = {'__name__': '__main__', '__file__': str(src_path)}
exec(compile('\n'.join(sanitized_lines) + '\n', str(src_path), 'exec'), exec_globals)
